### **4.1 Problema 1: Maximização de Lucro (Alocação de Frota)**

Contexto: Analisando a tabela Entregadores da base de dados da LogiPrime, operamos com dois modais principais na coluna Veiculo: Motorcycle (Motocicleta) e Scooter. A gerência precisa definir o limite diário de despachos para cada tipo de veículo visando maximizar o lucro, respeitando a jornada máxima de trabalho e regras de segurança no trânsito.

- **Motorcycle (x1):** lucro líquido de R$ 25,00 por pedido. Consome em média **0,5 horas**.
- **Scooter (x2):** lucro líquido de R$ 15,00 por pedido. Consome em média **0,75 horas**.

A operação dispõe de um máximo de **150 horas totais** de agentes por dia. Além disso, por diretrizes de segurança, o limite máximo de envios por **Motorcycle** não pode ultrapassar **200 entregas diárias**.

**Código Python (Google Colab / pulp):**

In [2]:
from pulp import LpMaximize, LpProblem, LpVariable
import sqlite3
import pandas as pd

conn = sqlite3.connect('meu_banco.db')

entregadores = pd.read_sql("SELECT Veiculo FROM 'logiPrime.Entregadores'", conn)

model = LpProblem(name="Max_Lucro_LogiPrime", sense=LpMaximize)

x1 = LpVariable(name="Qtd_Motorcycle", lowBound=0, cat="Integer")
x2 = LpVariable(name="Qtd_Scooter", lowBound=0, cat="Integer")

model += 25 * x1 + 15 * x2, "Lucro_Total"
model += (0.5 * x1 + 0.75 * x2 <= 150, "Restricao_Horas_Agentes")
model += (x1 <= 200, "Limite_Seguranca_Moto")

model.solve()

print(f"Status: {model.status}")
print(f"Entregas via Motorcycle: {x1.varValue}")
print(f"Entregas via Scooter: {x2.varValue}")
print(f"Lucro Máximo: R$ {model.objective.value():,.2f}")

Status: 1
Entregas via Motorcycle: 200.0
Entregas via Scooter: 66.0
Lucro Máximo: R$ 5,990.00


Análise de Cenário (What-If):




*   Cenário de Crise: E se as condições climáticas (coluna Weather = Stormy) piorarem e a gerência decidir reduzir drasticamente o número de entregas realizadas por caminhão devido aos riscos operacionais? Nesse caso, o limite de segurança cairia de 200 para apenas 80 entregas diárias.


*   Resultado Obtido: Alterando a restrição no código para caminhao <= 80 e executando novamente o modelo, o algoritmo adapta a operação instantaneamente. O sistema passa a recomendar o limite máximo permitido de 80 entregas por caminhão, respeitando as novas condições de segurança impostas pela tempestade. Dessa forma, apesar da redução na capacidade operacional e no lucro total, o modelo matemático garante a melhor alocação possível dos recursos disponíveis, mantendo a continuidade das operações mesmo em um cenário crítico.



## **4.2 Minimização de Custos (Distribuição Galpão x Destino)**



*   Capacidade máxima de transporte via caminhão: 1.050 pacotes diários.

*   Demanda total de entregas: 1.050 pacotes.

*  Custo operacional médio por entrega via caminhão: variável conforme rota e condições de operação.

**Código Python (Google Colab / pulp):**

In [3]:
from pulp import LpMinimize, LpProblem, LpVariable, lpSum

galpoes = ['Sao_Paulo', 'Campinas']
destinos = ['Santos', 'SJ_Campos', 'Ribeirao_Preto']

oferta = {'Sao_Paulo': 600, 'Campinas': 450}
demanda = {'Santos': 400, 'SJ_Campos': 350, 'Ribeirao_Preto': 300}

custos = {
    'Sao_Paulo': {'Santos': 5, 'SJ_Campos': 8, 'Ribeirao_Preto': 15},
    'Campinas': {'Santos': 12, 'SJ_Campos': 10, 'Ribeirao_Preto': 7}
}

model = LpProblem(name="Min_Custo_LogiPrime", sense=LpMinimize)

rotas = [(i, j) for i in galpoes for j in destinos]
x = LpVariable.dicts("Rota", rotas, lowBound=0, cat="Integer")

model += lpSum([custos[i][j] * x[(i, j)] for i in galpoes for j in destinos])

for i in galpoes:
    model += lpSum([x[(i, j)] for j in destinos]) <= oferta[i]

for j in destinos:
    model += lpSum([x[(i, j)] for i in galpoes]) == demanda[j]

model.solve()

print(f"Status: {model.status}")
for i in galpoes:
    for j in destinos:
        if x[(i, j)].varValue > 0:
            print(f"De {i} para {j}: {x[(i, j)].varValue} pacotes")

Status: 1
De Sao_Paulo para Santos: 400.0 pacotes
De Sao_Paulo para SJ_Campos: 200.0 pacotes
De Campinas para SJ_Campos: 150.0 pacotes
De Campinas para Ribeirao_Preto: 300.0 pacotes


Análise de Cenário (What-If):

*   Cenário de Crise: E se a base apontar trânsito severo (coluna `Traffic = Jam`), aumentando significativamente o custo operacional das entregas realizadas por caminhão? Nesse cenário, o custo por operação poderia subir de R$ 18,00 para um valor ainda maior devido ao aumento do tempo de deslocamento e do consumo de combustível.

*   Resultado Obtido: Alterando o valor do custo operacional no código, o modelo recalcula automaticamente o impacto financeiro da operação. Dessa forma, o sistema permite avaliar rapidamente como o aumento do trânsito influencia o custo total das entregas, auxiliando a empresa na tomada de decisões estratégicas para minimizar prejuízos e manter a eficiência logística mesmo em situações críticas.



## **Análise Crítica Exigida (Otimização)**

Relatório Técnico - Validação da Modelagem:

Os resultados matemáticos obtidos no Python para os modelos de maximização e minimização apresentam coerência com a realidade operacional da LogiPrime. Conforme identificado no escopo do projeto, a empresa enfrentava dificuldades relacionadas à ineficiência na gestão das entregas, realizando os envios apenas por ordem de chegada, sem critérios de otimização logística.

Com a aplicação da Pesquisa Operacional e da modelagem matemática no Python, foi possível estruturar um sistema capaz de analisar restrições operacionais, como limite de horas disponíveis, capacidade máxima de entregas e custos operacionais do caminhão. Dessa forma, o algoritmo determina automaticamente a melhor utilização dos recursos disponíveis, buscando maximizar o lucro e minimizar os custos da operação.

Além disso, a utilização da Análise de Cenário (*What-If*) mostrou-se fundamental para apoiar a gestão de riscos e a tomada de decisões estratégicas. Ao simular situações críticas extraídas das variáveis da operação — como condições climáticas severas (*Stormy*) e congestionamentos intensos (*Jam*) —, o modelo consegue recalcular instantaneamente os impactos operacionais e financeiros causados por essas alterações.

Com isso, a LogiPrime deixa de atuar apenas de forma reativa diante de imprevistos operacionais. Utilizando os scripts desenvolvidos, a empresa passa a ter a capacidade de inserir novas condições operacionais no sistema e obter rapidamente um plano matemático de contingência, permitindo reduzir perdas financeiras, melhorar a previsibilidade logística e garantir maior segurança operacional nas entregas realizadas por caminhão.